In [121]:
import pandas as pd

In [122]:
data = pd.read_json("News_Category_Dataset_v3.json", lines = True)

In [123]:
data

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22
...,...,...,...,...,...,...
209522,https://www.huffingtonpost.com/entry/rim-ceo-t...,RIM CEO Thorsten Heins' 'Significant' Plans Fo...,TECH,Verizon Wireless and AT&T are already promotin...,"Reuters, Reuters",2012-01-28
209523,https://www.huffingtonpost.com/entry/maria-sha...,Maria Sharapova Stunned By Victoria Azarenka I...,SPORTS,"Afterward, Azarenka, more effusive with the pr...",,2012-01-28
209524,https://www.huffingtonpost.com/entry/super-bow...,"Giants Over Patriots, Jets Over Colts Among M...",SPORTS,"Leading up to Super Bowl XLVI, the most talked...",,2012-01-28
209525,https://www.huffingtonpost.com/entry/aldon-smi...,Aldon Smith Arrested: 49ers Linebacker Busted ...,SPORTS,CORRECTION: An earlier version of this story i...,,2012-01-28


In [124]:
data.shape

(209527, 6)

In [125]:
data.columns

Index(['link', 'headline', 'category', 'short_description', 'authors', 'date'], dtype='str')

In [126]:
data = data['short_description'].head(50000)

In [127]:
data.duplicated().sum()

np.int64(666)

In [128]:
data.drop_duplicates(inplace=True)

In [129]:
data

0        Health experts said it is too early to predict...
1        He was subdued by passengers and crew when he ...
2        "Until you have a dog you don't understand wha...
3        "Accidentally put grown-up toothpaste on my to...
4        Amy Cooper accused investment firm Franklin Te...
                               ...                        
49995    The Texas elector wrote in a blog post that he...
49996                                         Please clap!
49997                     An open letter from a mannequin.
49998    "Sometimes doctors don't assign the right gend...
49999    Get up close with these images from HuffPost R...
Name: short_description, Length: 49334, dtype: str

In [130]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [131]:
tokenizer = Tokenizer(oov_token='<OOV>')

In [132]:
tokenizer.fit_on_texts(data)

In [133]:
data = tokenizer.texts_to_sequences(data)

In [134]:
input_sequences = []

for seq in data:
    if len(seq) < 2:
        continue
    for i in range(1, len(seq)):
        ngram = seq[:i+1]
        input_sequences.append(ngram)

In [135]:
input_sequences

[[118, 506],
 [118, 506, 15],
 [118, 506, 15, 12],
 [118, 506, 15, 12, 8],
 [118, 506, 15, 12, 8, 151],
 [118, 506, 15, 12, 8, 151, 539],
 [118, 506, 15, 12, 8, 151, 539, 3],
 [118, 506, 15, 12, 8, 151, 539, 3, 4706],
 [118, 506, 15, 12, 8, 151, 539, 3, 4706, 306],
 [118, 506, 15, 12, 8, 151, 539, 3, 4706, 306, 1859],
 [118, 506, 15, 12, 8, 151, 539, 3, 4706, 306, 1859, 67],
 [118, 506, 15, 12, 8, 151, 539, 3, 4706, 306, 1859, 67, 2324],
 [118, 506, 15, 12, 8, 151, 539, 3, 4706, 306, 1859, 67, 2324, 49],
 [118, 506, 15, 12, 8, 151, 539, 3, 4706, 306, 1859, 67, 2324, 49, 13],
 [118, 506, 15, 12, 8, 151, 539, 3, 4706, 306, 1859, 67, 2324, 49, 13, 2],
 [118,
  506,
  15,
  12,
  8,
  151,
  539,
  3,
  4706,
  306,
  1859,
  67,
  2324,
  49,
  13,
  2,
  21080],
 [118,
  506,
  15,
  12,
  8,
  151,
  539,
  3,
  4706,
  306,
  1859,
  67,
  2324,
  49,
  13,
  2,
  21080,
  214],
 [118,
  506,
  15,
  12,
  8,
  151,
  539,
  3,
  4706,
  306,
  1859,
  67,
  2324,
  49,
  13,
  2,
  21

In [136]:
max_len = max([len(i) for i in input_sequences])

In [137]:
input = pad_sequences(input_sequences, padding='pre', maxlen=max_len)

In [138]:
X = input[:,:-1]
Y = input[:,-1]

In [139]:
Y = np.array(Y)

In [140]:
from sklearn.model_selection import train_test_split

In [141]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [142]:
vocab_size = len(tokenizer.word_index)+1

In [143]:
max_len = X.shape[1]

In [144]:
model = Sequential([Embedding(vocab_size, 128, input_length=max_len), LSTM(256), Dense(vocab_size, activation = 'softmax')])

d:\GenerativeAI\Project\Next_Word\venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [145]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [146]:
model.build(input_shape=(None, max_len))

In [ ]:
model.fit(X_train, Y_train, epochs=10, validation_data=(X_test, Y_test))

Epoch 1/10
17216/17216 ━━━━━━━━━━━━━━━━━━━━ 4863s 282ms/step - accuracy: 0.2098 - loss: 5.1764 - val_accuracy: 0.1579 - val_loss: 6.6190
Epoch 2/10
17216/17216 ━━━━━━━━━━━━━━━━━━━━ 4357s 253ms/step - accuracy: 0.2309 - loss: 4.8381 - val_accuracy: 0.1545 - val_loss: 6.7802
Epoch 3/10
17216/17216 ━━━━━━━━━━━━━━━━━━━━ 6632s 385ms/step - accuracy: 0.2534 - loss: 4.5398 - val_accuracy: 0.1528 - val_loss: 6.9478
Epoch 4/10
17216/17216 ━━━━━━━━━━━━━━━━━━━━ 0s 251ms/step - accuracy: 0.2761 - loss: 4.2829

KeyboardInterrupt: 

In [148]:
max_len

77